In [ ]:
!pip install datasets
!pip install dateparser
!pip install dateparser_data
!pip install faiss-gpu

!python -m spacy download fr_core_news_sm
!python -m spacy download en_core_web_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 9.8 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2024.12.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which i

In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 35.2 MB/s eta 0:00:00


In [ ]:
from itertools import islice

from datasets import load_dataset
import unicodedata
import re
import pandas as pd
from dateparser.search import search_dates
from dateparser_data.settings import default_parsers
import numpy as np
import spacy
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
from joblib import Parallel, delayed
from sentence_transformers import SentenceTransformer
import faiss
import json
from tqdm import tqdm
import time

In [ ]:
def load_lang(lang: str, max_docs: int) -> list:
    """
    Load the dataset for a specific language and return a list of documents.
    :param lang: language code (e.g., 'fr' or 'en')
    :param max_docs: maximum number of documents to a load
    :return: list of documents
    """
    print(f"[{lang.upper()}] Chargement...")
    dataset_stream = load_dataset('miracl/miracl-corpus', lang, split='train', streaming=True, trust_remote_code=True)
    sample = islice(dataset_stream, max_docs) if max_docs > 0 else dataset_stream

    docs = []
    for doc in sample:
        docs.append({
            "docid": doc["docid"],
            "lang": lang,
            "title": doc["title"],
            "text": doc["text"]
        })
    return docs

In [ ]:
languages = ['fr', 'en']
max_docs_per_lang = 100000 # taille du dataset (x2 car français + anglais)

In [ ]:
results = Parallel(n_jobs=len(languages))(
        delayed(load_lang)(lang, max_docs_per_lang) for lang in languages
    )

In [ ]:
all_docs = [doc for lang_docs in results for doc in lang_docs]

In [ ]:
def clean_text(text: str) -> str:
    """
    Clean the input text by performing various preprocessing steps.
    :param text: the text to clean
    :return: cleaned text
    """

    # Unicode normalization
    text = unicodedata.normalize('NFKC', text)

    # Replacing typographical quotation marks with single quotation marks
    text = text.replace('“', '"').replace('”', '"').replace('«', '"').replace('»', '"').replace("’", "'")

    # Replace \n with a space (or a period + space if it breaks a sentence)
    text = text.replace('\n', '. ')

    # Delete invisible control characters (except those already managed)
    text = re.sub(r'[\x00-\x1F\x7F-\x9F]', '', text)

    # Delete multiple spaces
    text = text.replace('\\n', ' ').replace('\n', ' ')
    text = re.sub(r'\s+', ' ', text)

    # Delete multiple spaces around punctuation
    text = re.sub(r'\.{2,}', '.', text)
    text = re.sub(r'\?{2,}', '?', text)
    text = re.sub(r'\!{2,}', '!', text)

    # Delete spaces before punctuation
    text = re.sub(r'\s+([.,!?;:])', r'\1', text)
    text = re.sub(r'([.,!?;:])([^\s])', r'\1 \2', text)

    # Delete HTML tags and HTML entities
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'&\w+;', '', text)

    # Delete unprintable characters or orphan symbols
    text = re.sub(r'[^\x20-\x7EÀ-ÿ€£$¥•–—’“”…°²³µ·]', '', text)

    return text.strip()

In [ ]:
df = pd.DataFrame(all_docs)
df["title"] = df["title"].apply(clean_text)
df["text"] = df["text"].apply(clean_text)

In [ ]:
def chunking(df : pd.DataFrame, chunk_size:int) -> pd.DataFrame:
    """
    Chunking the dataframe into smaller chunks of a specified size.
    :param df: the dataframe to chunk
    :param chunk_size: the size of each chunk
    :return: a new dataframe with the chunks
    """

    # TODO : chunking by chunk_size

    chunk_text = []
    chunk_meta = []

    for index, row in df.iterrows():
        text = row["text"]
        title = row["title"]
        docid = row["docid"]
        lang = row["lang"]
        sentences = text.split(". ")

        for sentence in sentences:

            chunk_text.append(sentence)
            chunk_meta.append({
                "title": title,
                "docid": docid,
                "lang": lang
            })

    if len(chunk_text) != len(chunk_meta):
        raise ValueError("Chunk text and metadata lengths do not match.")

    df_chunk = pd.DataFrame({"text": chunk_text, "meta": chunk_meta})
    df_chunk = df_chunk.drop_duplicates(subset=["text"])
    df_chunk = df_chunk.dropna(subset=["text"])

    return df_chunk

In [ ]:
df_chunk = chunking(df, 0)

In [ ]:
def contains_explicit_1_january(text: str) -> bool:
    """
    Check if the text contains an explicit mention of 1st January.
    :param text: the text to check
    :return: True if the text contains an explicit mention of 1st January, False otherwise
    """
    patterns = [
        r"\b0?1[\/\-\. ]?0?1\b",  # 01/01, 1/1, 01-01, etc.
        r"\b(1er|1|01)[^\d]?(janvier|january)\b",  # 1 janvier, 1er janvier, 01 janvier
        r"\b(janvier|january)[^\d]*(1er|1|01)\b",  # janvier 1, january 1st
        r"\b(1st|first) of (january|janvier)\b"  # 1st of January
    ]
    for pattern in patterns:
        if re.search(pattern, text, flags=re.IGNORECASE):
            return True
    return False


def has_explicit_date_pattern(text: str) -> bool:
    """
    Check if the text contains an explicit date pattern. In case of ambiguous date.
    :param text: the text to check
    :return: True if the text contains an explicit date pattern, False otherwise
    """
    numeric_patterns = [
        r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b",
        r"\b\d{4}\b"
    ]

    textual_patterns = [
        r"\b\d{1,2}\s+(janvier|février|mars|avril|mai|juin|juillet|août|septembre|octobre|novembre|décembre|"
        r"january|february|march|april|may|june|july|august|september|october|november|december)\s+\d{2,4}\b"
    ]

    all_patterns = numeric_patterns + textual_patterns

    for pattern in all_patterns:
        if re.search(pattern, text, flags=re.IGNORECASE):
            return True
    return False


def split_on_conjunctions(text: str) -> list:
    """
    Split the text on conjunctions (et, and, , or ;).
    :param text: the text to split
    :return: the list of segments
    """

    text = re.sub(r"\([^)]*\)", "", text)

    return re.split(r"\s+(et|and|,|;)\s+", text)


def is_pure_date_expression(text: str) -> bool:
    """
    Check if the text is a pure date expression.
    :param text: the text to check
    :return: True if the text is a pure date expression, False otherwise
    """
    patterns = [
        r"\b\d{1,2}[/-]\d{1,2}([/-]\d{2,4})?\b",  # 15/04[/2025]
        r"\b\d{1,2}\s+(janvier|février|mars|avril|mai|juin|juillet|août|septembre|octobre|novembre|décembre)\b",
        r"\b(january|february|march|april|may|june|july|august|september|october|november|december)\s+\d{1,2}\b"
    ]
    return any(re.search(p, text, flags=re.IGNORECASE) for p in patterns)


def extract_date(df: pd.DataFrame) -> pd.DataFrame:
    parsers = [parser for parser in default_parsers if parser != 'relative-time']

    for i, row in df.iterrows():
        lang = row["meta"]["lang"]
        res_list = []

        try:
            # On divise le texte sur les conjonctions
            segments = [seg.strip() for seg in split_on_conjunctions(row["text"]) if
                        seg.strip().lower() not in ["et", "and", ",", ";"] and seg.strip() != ""]

            for segment in segments:
                dates_found = search_dates(
                    segment,
                    languages=["fr", "en"],
                    settings={
                        'PREFER_DAY_OF_MONTH': 'first',
                        'PREFER_MONTH_OF_YEAR': 'first',
                        'DATE_ORDER': 'DMY' if lang == 'fr' else 'MDY',
                        'PARSERS': parsers,
                    }
                )

                if dates_found:
                    for matched_text, result in dates_found:
                        res = {}

                        if result.day == 1 and result.month == 1:
                            if contains_explicit_1_january(matched_text):
                                res["year"] = result.year if has_explicit_date_pattern(matched_text) else None
                                res["month"] = result.month
                                res["day"] = result.day
                            else:
                                res["year"] = result.year if has_explicit_date_pattern(matched_text) else None
                                res["month"] = None
                                res["day"] = None
                        else:
                            res["year"] = result.year if has_explicit_date_pattern(matched_text) else None
                            res["month"] = result.month
                            res["day"] = result.day

                        res["hour"] = result.hour if result.hour else None
                        res["minute"] = result.minute if result.minute else None
                        res["second"] = result.second if result.second else None

                        # False positive check
                        if res["day"] and res["month"] and not is_pure_date_expression(matched_text):
                            break

                        if res["day"] or res["month"] or res["year"]:
                            res_list.append(res)

        except Exception as e:
            res_list = [{"error": str(e)}]

        df.at[i, "meta"]["dates"] = res_list

    return df

In [ ]:
df_chunk = extract_date(df_chunk)

In [ ]:
chunks = np.array_split(df_chunk, 8)

/usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [ ]:
_nlp_models = {}

def get_nlp(lang : str) -> spacy.Language:
    """
    Load the spaCy model for the given language if not already loaded.
    :param lang: the language to load the model for
    :return: the spaCy model for the given language
    """
    if lang not in _nlp_models:
        _nlp_models[lang] = spacy.load(
            "fr_core_news_sm" if lang == "fr" else "en_core_web_sm",
        )
    return _nlp_models[lang]

def post_treatment_bert_entities(entities: list[tuple[str, str]]) -> list[tuple[str, str]]:
    """
    Post-process the entities detected by BERT to clean them up before sending them to spaCy.
    :param entities: the list of entities detected by BERT
    :return: the cleaned list of entities
    """
    cleaned = []
    current = ""
    label = None

    for word, tag in entities:
        if word.startswith("##"):
            current += word[2:]
        else:
            if current:
                cleaned.append((current.strip(), label))
            current = word
            label = tag

    if current:
        cleaned.append((current.strip(), label))

    cleaned = [(w, t) for w, t in cleaned if len(w) > 2 and w.lower() not in {"l'", "’", "le", "la"}]

    return cleaned

def enrich_df_with_ner_pipe(df_chunk: pd.DataFrame) -> pd.DataFrame:
    """
    Enrich the DataFrame with Named Entity Recognition (NER) using BERT and spaCy.
    :param df_chunk: the DataFrame to enrich
    :return: the enriched DataFrame
    """
    tokenizer = AutoTokenizer.from_pretrained("dslim/bert-large-NER")
    model = AutoModelForTokenClassification.from_pretrained("dslim/bert-large-NER")
    bert_ner = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

    entities_all = []

    for index, row in df_chunk.iterrows():

        text = row["text"]
        lang = row["meta"]["lang"]

        # BERT NER
        ents = [(ent['word'], ent['entity_group']) for ent in bert_ner(text)]
        ents = post_treatment_bert_entities(ents)

        final_ents = []
        for ent_text, ent_label in ents:
            if ent_label not in ("None", "DATE"):
              if ent_label == "MISC":
                  spacy_nlp = get_nlp(lang)
                  doc = spacy_nlp(ent_text)
                  sub_ents = [(e.text, e.label_) for e in doc.ents]
                  if sub_ents:
                      final_ents.extend(sub_ents)
              else:
                  final_ents.append((ent_text, ent_label))

        entities_all.append(final_ents)

    df_chunk = df_chunk.copy()
    df_chunk["entities"] = entities_all
    return df_chunk

In [ ]:
df_copy = df_chunk.copy()

In [ ]:
df_chunk = df_copy

In [ ]:
results = enrich_df_with_ner_pipe(df_chunk)

tokenizer_config.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.45k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Some weights of the model checkpoint at dslim/bert-large-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [ ]:
results

,text,meta,entities
0,"Paul Jules Antoine Meillet, né le à Moulins (A...","{'title': 'Antoine Meillet', 'docid': '3#0', '...","[(Paul Jules Antoine Meillet, PER), (Moulins, ..."
1,Il est aussi philologue.,"{'title': 'Antoine Meillet', 'docid': '3#0', '...",[]
2,"D'origine bourbonnaise, fils d'un notaire de C...","{'title': 'Antoine Meillet', 'docid': '3#1', '...","[(naise, None), (Châteaumeillant, LOC), (Cher,..."
3,Étudiant à la faculté des lettres de Paris à p...,"{'title': 'Antoine Meillet', 'docid': '3#2', '...","[(é des lettres de, None), (Paris, LOC), (Loui..."
4,"En 1889, il est major de l'agrégation de gramm...","{'title': 'Antoine Meillet', 'docid': '3#3', '...",[]
...,...,...,...
838,Although these theories lack convincing scient...,"{'title': 'Autism', 'docid': '25#16', 'lang': ...",[]
839,Autism's symptoms result from maturation-relat...,"{'title': 'Autism', 'docid': '25#17', 'lang': ...",[]
840,How autism occurs is not well understood,"{'title': 'Autism', 'docid': '25#17', 'lang': ...",[]
841,Its mechanism can be divided into two areas: t...,"{'title': 'Autism', 'docid': '25#17', 'lang': ...",[]


In [ ]:
models = {
    "all-MiniLM-L6-v2": "sentence-transformers/all-MiniLM-L6-v2",
    "e5-small-v2": "intfloat/e5-small-v2",
    "multilingual-e5-large-instruct": "intfloat/multilingual-e5-large-instruct",
    "linq-embed-mistral": "Linq-AI-Research/Linq-Embed-Mistral",
    "SFR-Embedding-Mistral": "Salesforce/SFR-Embedding-Mistral"
}

In [ ]:
def evaluate_model_massive(model_name, model_path, corpus, meta, k=5):
    print(f"\nEvaluating {model_name} on massive dataset...")
    model = SentenceTransformer(model_path)

    start_time = time.time()
    doc_embeddings = model.encode(corpus, batch_size=64, normalize_embeddings=True, convert_to_numpy=True)
    time_docs = time.time() - start_time

    dim = model.get_sentence_embedding_dimension()
    index = faiss.IndexFlatIP(dim)
    index.add(doc_embeddings)

    # Similarity analysis
    similarities_top1 = []
    similarities_top5 = []
    diff_top1_top2 = []

    batch_size = 5000
    for i in range(0, len(doc_embeddings), batch_size):
        batch_embs = doc_embeddings[i:i+batch_size]
        D, I = index.search(batch_embs, k)

        for d in D:
            similarities_top1.append(d[0])
            similarities_top5.append(np.mean(d[:5]))
            if k >= 2:
                diff_top1_top2.append(d[0] - d[1])

    corpus_langs = [m['lang'] for m in meta]

    results = {
        "Model": model_name,
        "Mean Top-1 Similarity": round(np.mean(similarities_top1), 4),
        "Mean Top-5 Similarity": round(np.mean(similarities_top5), 4),
        "Mean Drop Top-1 to Top-2": round(np.mean(diff_top1_top2), 4),
        "Doc Encoding Time (s)": round(time_docs, 2),
        "Embedding Dimension": dim,
        "Corpus Languages": list(set(corpus_langs))
    }
    return results

In [ ]:
all_results = []

In [ ]:
corpus = df_chunk["text"].tolist()
meta = df_chunk["meta"].tolist()

In [ ]:
def run_full_benchmark_massive():
    all_results = []

    for model_name, model_path in models.items():
        res = evaluate_model_massive(model_name, model_path, corpus, meta)
        all_results.append(res)

    df_results = pd.DataFrame(all_results)
    print("\nBenchmark finished!")
    print(df_results)
    df_results.to_csv('./benchmark_results_massive.csv', index=False)
    print("\nResults saved to ./benchmark_results_massive.csv")

In [ ]:
run_full_benchmark()

In [ ]:
# tests